
# Knowledge Graph Embeddings con TransE
---

Cálculo de similitud entre películas usando embeddings de grafos de conocimiento

In [1]:

# ============================================================================
# PARTE 1: INSTALACIÓN Y CONFIGURACIÓN
# ============================================================================
"""
Instalar las librerías necesarias:
pip install pykeen rdflib numpy scikit-learn matplotlib torch
"""
import numpy as np
from typing import List, Tuple, Dict
import torch
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from rdflib import Graph, Namespace, RDF, RDFS, Literal
import sbc_tools as sbc


d:\Users\pedro\Documents\Programacion\Master\RRBC\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# ============================================================================
# PARTE 2: CREACIÓN DEL GRAFO DE CONOCIMIENTO DE PELÍCULAS
# ============================================================================

def crear_grafo_peliculas_rdf():
    """
    Crea un grafo RDF de ejemplo con películas, géneros, directores y actores
    Similar a lo que genera la clase FilmGenreOntology de la práctica anterior
    """
    g = Graph()
    
    # Definir namespaces
    EX = Namespace("http://example.org/movies#")
    SCHEMA = Namespace("http://schema.org/")
    
    g.bind("ex", EX)
    g.bind("schema", SCHEMA)
    
    # Películas de ciencia ficción
    g.add((EX.Inception, RDF.type, SCHEMA.Movie))
    g.add((EX.Inception, SCHEMA.name, Literal("Inception")))
    g.add((EX.Inception, SCHEMA.genre, EX.SciFi))
    g.add((EX.Inception, SCHEMA.genre, EX.Thriller))
    g.add((EX.Inception, SCHEMA.director, EX.ChristopherNolan))
    g.add((EX.Inception, SCHEMA.actor, EX.LeonardoDiCaprio))
    
    g.add((EX.Interstellar, RDF.type, SCHEMA.Movie))
    g.add((EX.Interstellar, SCHEMA.name, Literal("Interstellar")))
    g.add((EX.Interstellar, SCHEMA.genre, EX.SciFi))
    g.add((EX.Interstellar, SCHEMA.genre, EX.Drama))
    g.add((EX.Interstellar, SCHEMA.director, EX.ChristopherNolan))
    g.add((EX.Interstellar, SCHEMA.actor, EX.MatthewMcConaughey))
    
    g.add((EX.Matrix, RDF.type, SCHEMA.Movie))
    g.add((EX.Matrix, SCHEMA.name, Literal("The Matrix")))
    g.add((EX.Matrix, SCHEMA.genre, EX.SciFi))
    g.add((EX.Matrix, SCHEMA.genre, EX.Action))
    g.add((EX.Matrix, SCHEMA.director, EX.Wachowskis))
    g.add((EX.Matrix, SCHEMA.actor, EX.KeanuReeves))
    
    # Películas de drama
    g.add((EX.Titanic, RDF.type, SCHEMA.Movie))
    g.add((EX.Titanic, SCHEMA.name, Literal("Titanic")))
    g.add((EX.Titanic, SCHEMA.genre, EX.Drama))
    g.add((EX.Titanic, SCHEMA.genre, EX.Romance))
    g.add((EX.Titanic, SCHEMA.director, EX.JamesCameron))
    g.add((EX.Titanic, SCHEMA.actor, EX.LeonardoDiCaprio))
    
    g.add((EX.ShawshankRedemption, RDF.type, SCHEMA.Movie))
    g.add((EX.ShawshankRedemption, SCHEMA.name, Literal("The Shawshank Redemption")))
    g.add((EX.ShawshankRedemption, SCHEMA.genre, EX.Drama))
    g.add((EX.ShawshankRedemption, SCHEMA.director, EX.FrankDarabont))
    g.add((EX.ShawshankRedemption, SCHEMA.actor, EX.TimRobbins))
    
    # Películas de acción
    g.add((EX.JohnWick, RDF.type, SCHEMA.Movie))
    g.add((EX.JohnWick, SCHEMA.name, Literal("John Wick")))
    g.add((EX.JohnWick, SCHEMA.genre, EX.Action))
    g.add((EX.JohnWick, SCHEMA.genre, EX.Thriller))
    g.add((EX.JohnWick, SCHEMA.director, EX.ChadStahelski))
    g.add((EX.JohnWick, SCHEMA.actor, EX.KeanuReeves))
    
    g.add((EX.DarkKnight, RDF.type, SCHEMA.Movie))
    g.add((EX.DarkKnight, SCHEMA.name, Literal("The Dark Knight")))
    g.add((EX.DarkKnight, SCHEMA.genre, EX.Action))
    g.add((EX.DarkKnight, SCHEMA.genre, EX.Crime))
    g.add((EX.DarkKnight, SCHEMA.director, EX.ChristopherNolan))
    g.add((EX.DarkKnight, SCHEMA.actor, EX.ChristianBale))
    
    # Relaciones adicionales entre entidades
    g.add((EX.SciFi, RDFS.label, Literal("Science Fiction")))
    g.add((EX.Drama, RDFS.label, Literal("Drama")))
    g.add((EX.Action, RDFS.label, Literal("Action")))
    g.add((EX.Thriller, RDFS.label, Literal("Thriller")))
    
    return g

def rdf_a_triples(g: Graph) -> List[Tuple[str, str, str]]:
    """
    Convierte un grafo RDF a formato de triples para PyKEEN
    Formato: (sujeto, predicado, objeto)
    """
    triples = []
    for s, p, o in g:
        # Convertir URIs y literales a strings simplificados
        subj = str(s).split('#')[-1] if '#' in str(s) else str(s).split('/')[-1]
        pred = str(p).split('#')[-1] if '#' in str(p) else str(p).split('/')[-1]
        obj = str(o).split('#')[-1] if '#' in str(o) else str(o).split('/')[-1]
        
        # Filtrar literales de nombres (opcional, para simplificar)
        if not obj.startswith('"'):
            triples.append((subj, pred, obj))
    
    return triples


In [3]:

# ============================================================================
# PARTE 3: ENTRENAMIENTO DEL MODELO TransE
# ============================================================================

def entrenar_transe(triples: List[Tuple[str, str, str]], 
                    embedding_dim: int = 50, 
                    num_epochs: int = 100):
    """
    Entrena un modelo TransE sobre el grafo de conocimiento
    
    TransE representa las relaciones como traslaciones en el espacio vectorial:
    h + r ≈ t  (cabeza + relación ≈ cola)
    
    Parámetros:
    - embedding_dim: dimensión de los vectores de embedding
    - num_epochs: número de épocas de entrenamiento
    """
    print("=" * 70)
    print("ENTRENANDO MODELO TransE")
    print("=" * 70)
    
    # Crear TriplesFactory desde la lista de triples
    tf = TriplesFactory.from_labeled_triples(
        np.array(triples, dtype=str),
        create_inverse_triples=True  # Añade triples inversos (mejora resultados)
    )
    
    print(f"\nEstadísticas del grafo:")
    print(f"  - Número de entidades: {tf.num_entities}")
    print(f"  - Número de relaciones: {tf.num_relations}")
    print(f"  - Número de triples: {tf.num_triples}")
    
    # Configurar y entrenar el pipeline de TransE
    result = pipeline(
        model='TransE',
        training=tf,
        testing=tf,  # En aplicación real, separar train/test
        model_kwargs={
            'embedding_dim': embedding_dim,
            'scoring_fct_norm': 2  # Norma L2
        },
        optimizer='Adam',
        optimizer_kwargs={
            'lr': 0.01
        },
        training_kwargs={
            'num_epochs': num_epochs,
            'batch_size': 32
        },
        random_seed=42,
        device='cpu'
    )
    
    print(f"\n✓ Modelo entrenado exitosamente")
    print(f"  - Loss final: {result.losses[-1]:.4f}")
    
    return result


In [4]:

# ============================================================================
# PARTE 4: CÁLCULO DE SIMILITUD ENTRE PELÍCULAS
# ============================================================================

def obtener_embedding_entidad(model, entity_name: str, entity_to_id: Dict) -> np.ndarray:
    """
    Obtiene el vector embedding de una entidad
    """
    if entity_name not in entity_to_id:
        raise ValueError(f"Entidad '{entity_name}' no encontrada en el grafo")
    
    entity_id = entity_to_id[entity_name]
    embedding = model.entity_representations[0](
        torch.tensor([entity_id])
    ).detach().numpy()[0]
    
    return embedding

def calcular_similitudes_peliculas(result, peliculas: List[str]):
    """
    Calcula la matriz de similitud entre películas usando embeddings TransE
    """
    model = result.model
    entity_to_id = result.training.entity_to_id
    
    print("\n" + "=" * 70)
    print("CÁLCULO DE SIMILITUDES CON EMBEDDINGS TransE")
    print("=" * 70)
    
    # Obtener embeddings de todas las películas
    embeddings = []
    peliculas_validas = []
    
    for pelicula in peliculas:
        try:
            emb = obtener_embedding_entidad(model, pelicula, entity_to_id)
            embeddings.append(emb)
            peliculas_validas.append(pelicula)
        except ValueError as e:
            print(f"⚠ {e}")
    
    embeddings = np.array(embeddings)
    
    # Calcular matriz de similitud coseno
    sim_matrix = cosine_similarity(embeddings)
    
    # Mostrar resultados
    print(f"\nMatriz de similitud (coseno) entre {len(peliculas_validas)} películas:\n")
    print(f"{'':20}", end='')
    for p in peliculas_validas:
        print(f"{p:12}", end='')
    print()
    
    for i, p1 in enumerate(peliculas_validas):
        print(f"{p1:20}", end='')
        for j, p2 in enumerate(peliculas_validas):
            print(f"{sim_matrix[i][j]:12.4f}", end='')
        print()
    
    return sim_matrix, peliculas_validas, embeddings

def encontrar_peliculas_similares(result, pelicula_query: str, top_k: int = 3):
    """
    Encuentra las k películas más similares a una película dada
    """
    model = result.model
    entity_to_id = result.training.entity_to_id
    id_to_entity = {v: k for k, v in entity_to_id.items()}
    
    # Obtener todas las películas del grafo
    todas_peliculas = [ent for ent in entity_to_id.keys() 
                       if ent not in ['type', 'Movie', 'director', 'actor', 'genre', 
                                     'name', 'label', 'SciFi', 'Drama', 'Action', 
                                     'Thriller', 'Romance', 'Crime']]
    
    # Obtener embedding de la película query
    try:
        query_emb = obtener_embedding_entidad(model, pelicula_query, entity_to_id)
    except ValueError as e:
        print(f"Error: {e}")
        return
    
    # Calcular similitud con todas las películas
    similitudes = []
    for pelicula in todas_peliculas:
        if pelicula != pelicula_query:
            try:
                emb = obtener_embedding_entidad(model, pelicula, entity_to_id)
                sim = cosine_similarity([query_emb], [emb])[0][0]
                similitudes.append((pelicula, sim))
            except:
                pass
    
    # Ordenar por similitud
    similitudes.sort(key=lambda x: x[1], reverse=True)
    
    print(f"\n{'=' * 70}")
    print(f"Top {top_k} películas similares a '{pelicula_query}':")
    print(f"{'=' * 70}")
    for i, (pelicula, sim) in enumerate(similitudes[:top_k], 1):
        print(f"{i}. {pelicula:20} (similitud: {sim:.4f})")


In [ ]:
# 1. Crear grafo de conocimiento
print("\nCreando grafo de conocimiento de películas...")
grafo_rdf = crear_grafo_peliculas_rdf()
triples = rdf_a_triples(grafo_rdf)
print(f"✓ Grafo creado con {len(triples)} triples")


In [ ]:

# 2. Entrenar modelo TransE
print("\nEntrenando modelo TransE...")
resultado = entrenar_transe(triples, embedding_dim=50, num_epochs=100)


In [ ]:

# 3. Calcular similitudes
print("\nCalculando similitudes entre películas...")
peliculas_interes = [
    'Inception', 'Interstellar', 'Matrix', 'Titanic', 
    'ShawshankRedemption', 'JohnWick', 'DarkKnight'
]
sim_matrix, peliculas_validas, embeddings = calcular_similitudes_peliculas(
    resultado, peliculas_interes
)


In [ ]:

# 4. Encontrar películas similares
print("\nBuscando películas similares...")
encontrar_peliculas_similares(resultado, 'Inception', top_k=3)
encontrar_peliculas_similares(resultado, 'Matrix', top_k=3)

    

    


---
# Ejemplo con ontología completa

In [ ]:
graph = sbc.load("ontologia_generos_cinematograficos_completa.ttl", format="turtle")

<Graph identifier=N9ec9e5c473e94bbbae489b69de9ee939 (<class 'rdflib.graph.Graph'>)>

In [ ]:
# 1. Crear grafo de conocimiento
print("\nCreando grafo de conocimiento de películas...")
triples = rdf_a_triples(graph)
print(f"✓ Grafo creado con {len(triples)} triples")


In [ ]:

# 2. Entrenar modelo TransE
print("\nEntrenando modelo TransE...")
resultado = entrenar_transe(triples, embedding_dim=50, num_epochs=100)


In [ ]:

# 3. Calcular similitudes
print("\nCalculando similitudes entre películas...")
peliculas_interes = ['Q24953', 'Q106665587', 'Q930622', 'Q135619158', 'Q121840231', 'Q426332', 'Q2364466', 'Q943961', 'Q108543', 'Q211009', 'Q18342910', 'Q20653259', 'Q15808646']

sim_matrix, peliculas_validas, embeddings = calcular_similitudes_peliculas(
    resultado, peliculas_interes
)


In [ ]:

# 4. Encontrar películas similares
print("\nBuscando películas similares...")
encontrar_peliculas_similares(resultado, 'Q106665587', top_k=3)
encontrar_peliculas_similares(resultado, 'Q930622', top_k=3)

In [11]:
from pykeen.checkpoints import save_model
resultado.save_to_directory("graph_embeddings")

INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=13235, num_relations=16, create_inverse_triples=True, num_triples=16501) to file:///Users/juanan/docencia/2025-2026/SBC/practicas-sbc/graph_embeddings/training_triples
INFO:pykeen.pipeline.api:Saved to directory: /Users/juanan/docencia/2025-2026/SBC/practicas-sbc/graph_embeddings


---
# Ejercicio

## Aplica las métricas de similitud basada en embeddings a tu grafo de conocimiento.

## Analiza los resultados y aplicabilidad

## Compara con los resultados de las métricas de similitud estructural

## Prueba otros métodos de la libreíra PyKeen (TransD, TransH, ...) Analiza los resultados

In [4]:
import numpy as np
from typing import List, Tuple, Dict
import torch
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from rdflib import Graph, Namespace, RDF, RDFS, Literal
import sbc_tools as sbc

Este modelo tiene su explicación, el modelo TransE creado anteriormente nos funcionaba correctamente pero las respuestas que daba no nos convencían del todo. Por ejemplo, daba similitudes muy altas a todos los animes (entre 0.86 y 0.97) algo raro para la gran diversidad de animes que tenemos, y luego tras usar el k_near para los dos ejemplos completos, ya nos convencimos de que algo estaba mal, porque daba resultados raros, por ejemplo, los más simiales a One Piece (anime de aventuras y piratas y shonen) eran Yakitate_Japan (comedia/gastronomía), Full_Metal_Panic Fumoffu (mechas/ comedia) y Full_Metal_Panic (mechas/comedia).

Tras una breve búsqueda por la ontología descubrimos que se debía a que, la similitud encontrada, era por los actores de voz, ya que compartian muchos de ellos. Tener una ontología tan rica en datos y relaciones, nos jugó en contra a la hora de hacer las similitudes. Esto obviamente es correcto, pero no lo que queríamos, por lo buscamos la manera de darle más peso a los géneros (multiplicando las tripletas de género) y conseguimos que nos diera resultados mejores, como los que se comprueban abajo.

Ahora los animes más similares a One Piece (piratas/aventura/shonen) son One Piece":
- Rurouni Kenshin — aventura, shonen
- Beck — (probablemente ha salido por los personajes,) 
- Berserk — dark fantasy, adventure, número de episodios, Epic


Y para Cowboy Bebop(sci-fi espacial, tono serio, rollo punk):
- Gundam (Sci-fi espacial, tono serio, mecha)
- Mobile Suit Victory Gundam (Sci-fi espacial)
- Monster (esta si puede generar duda, aunque ambos animes tienen un Target de jóvenes adultos)

El modelo no es perfecto como vemos, pero mejora significativamente los resultados anteriores

In [ ]:
def entrenar_transe_own(triples: List[Tuple[str, str, str]], 
                    embedding_dim: int = 128,  
                    num_epochs: int = 300):   
    
    print("=" * 70)
    print("PRE-PROCESSING DATA WEIGHTS")
    
    final_triples = []
    for h, r, t in triples:
        final_triples.append((h, r, t))
        if 'type' in r or 'Genre' in t or 'genre' in r:
            for _ in range(10): 
                final_triples.append((h, r, t))
                
    print(f"Triples original: {len(triples)} -> Weighted: {len(final_triples)}")


    tf = TriplesFactory.from_labeled_triples(
        np.array(final_triples, dtype=str),
        create_inverse_triples=True
    )
    

    result = pipeline(
        model='TransE',
        training=tf,
        testing=tf,
        model_kwargs={
            'embedding_dim': embedding_dim,
            'scoring_fct_norm': 1  
        },
        optimizer='Adam',
        optimizer_kwargs={'lr': 0.001}, 
        
        negative_sampler='bernoulli',
        
        training_kwargs={
            'num_epochs': num_epochs,
            'batch_size': 64
        },
        random_seed=42,
        device='cpu'
    )
    
    return result

In [9]:
g = sbc.load("ontologiaAnimesIncompleta.ttl")

tripletas = rdf_a_triples(g)


In [ ]:
import os
import torch
from pykeen.pipeline import pipeline

save_dir = "graph_embeddings"
model_path = os.path.join(save_dir, "trained_model.pkl")
factory_path = os.path.join(save_dir, "training_triples")


if os.path.exists(model_path):
    print("Found saved model. Loading...")
    
    model = torch.load(
                model_path, 
                map_location=torch.device('cpu'), 
                weights_only=False 
            )
    training_factory = TriplesFactory.from_path_binary(factory_path)

    class ModelWrapper:
        def __init__(self, model, training_factory):
            self.model = model
            self.training = training_factory

    resultado = ModelWrapper(model, training_factory)
    
else:
    
    resultado = entrenar_transe_own(tripletas, embedding_dim=50, num_epochs=100)
    
    resultado.save_to_directory(save_dir)


Found saved model. Loading...


In [11]:

# 3. Calcular similitudes
print("\nCalculando similitudes entre animes...")
animes_interes = ['Cowboy_Bebop', 'Rurouni_Kenshin', 'Oh_My_Goddess', 'Angel_Sanctuary', 'Zipang', 'Neon_Genesis_Evangelion_The_End_of_Evangelion', 'Initial_D_Fourth_Stage']

sim_matrix, animes_validos, embeddings = calcular_similitudes_peliculas(
    resultado, animes_interes
)



Calculando similitudes entre animes...

CÁLCULO DE SIMILITUDES CON EMBEDDINGS TransE

Matriz de similitud (coseno) entre 7 películas:

                    Cowboy_BebopRurouni_KenshinOh_My_GoddessAngel_SanctuaryZipang      Neon_Genesis_Evangelion_The_End_of_EvangelionInitial_D_Fourth_Stage
Cowboy_Bebop              1.0000      0.7526      0.6897      0.5573      0.6815      0.5370      0.5355
Rurouni_Kenshin           0.7526      1.0000      0.7815      0.4674      0.5735      0.6098      0.5050
Oh_My_Goddess             0.6897      0.7815      1.0000      0.5251      0.5578      0.6484      0.4412
Angel_Sanctuary           0.5573      0.4674      0.5251      1.0000      0.4981      0.4493      0.2750
Zipang                    0.6815      0.5735      0.5578      0.4981      1.0000      0.4171      0.3613
Neon_Genesis_Evangelion_The_End_of_Evangelion      0.5370      0.6098      0.6484      0.4493      0.4171      1.0000      0.4309
Initial_D_Fourth_Stage      0.5355      0.5050      0.

In [12]:

# 4. Encontrar películas similares
print("\nBuscando animes similares...")
encontrar_peliculas_similares(resultado, 'Cowboy_Bebop', top_k=3)
encontrar_peliculas_similares(resultado, 'One_Piece', top_k=3)


Buscando animes similares...

Top 3 películas similares a 'Cowboy_Bebop':
1. _Gundam              (similitud: 0.8206)
2. Mobile_Suit_Victory_Gundam (similitud: 0.8146)
3. Monster              (similitud: 0.8048)

Top 3 películas similares a 'One_Piece':
1. Rurouni_Kenshin      (similitud: 0.8223)
2. Beck_Mongolian_Chop_Squad (similitud: 0.8157)
3. Berserk              (similitud: 0.7926)


In [ ]:
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
import numpy as np
import torch
from typing import List, Tuple

def entrenar_transh(triples: List[Tuple[str, str, str]], 
                              embedding_dim: int = 128,
                              num_epochs: int = 300,
                              genre_weight: int = 10):


    tf_all = TriplesFactory.from_labeled_triples(
        np.array(triples, dtype=str),
        create_inverse_triples=False  
    )
    

    tf_train, tf_test = tf_all.split([0.8, 0.2], random_state=42)
    
    tf_train = tf_train.with_inverse_triples()
    

    genre_relation_ids = [
        id for label, id in tf_train.relation_to_id.items()
        if ('type' in label or 'Genre' in label) and not label.endswith('_inverse')
    ]
    
    print(f"Identified {len(genre_relation_ids)} relation types for weighting.")

    train_tensor = tf_train.mapped_triples
    

    is_genre_mask = torch.isin(train_tensor[:, 1], torch.tensor(genre_relation_ids))
    genre_triples = train_tensor[is_genre_mask]
    
    if len(genre_triples) > 0:

        repeated_genres = genre_triples.repeat(genre_weight - 1, 1)
        new_train_tensor = torch.cat([train_tensor, repeated_genres], dim=0)
    else:
        new_train_tensor = train_tensor

   
    tf_train_weighted = TriplesFactory(
        mapped_triples=new_train_tensor,
        entity_to_id=tf_train.entity_to_id,
        relation_to_id=tf_train.relation_to_id,
        create_inverse_triples=False,
    )

    print(f"Training Triples: {tf_train.num_triples} -> {tf_train_weighted.num_triples} (Weighted)")

    result = pipeline(
        model='TransH',
        training=tf_train_weighted,
        testing=tf_test,
        model_kwargs={
            'embedding_dim': embedding_dim,
            'scoring_fct_norm': 2  
        },
        optimizer='Adam',
        optimizer_kwargs={'lr': 0.005}, 
        negative_sampler='bernoulli',
        negative_sampler_kwargs={'num_negs_per_pos': 50}, 
        training_kwargs={
            'num_epochs': num_epochs,
            'batch_size': 256
        },
        device='cpu',
        random_seed=42
    )
    
    return result

In [14]:
import os
import torch
from pykeen.pipeline import pipeline

save_dir = "graph_embeddings_transh"
model_path = os.path.join(save_dir, "trained_model.pkl")
factory_path = os.path.join(save_dir, "training_triples")


if os.path.exists(model_path):
    print("Found saved model. Loading...")
    
    model = torch.load(
                model_path, 
                map_location=torch.device('cpu'), 
                weights_only=False 
            )
    training_factory = TriplesFactory.from_path_binary(factory_path)

    class ModelWrapper:
        def __init__(self, model, training_factory):
            self.model = model
            self.training = training_factory

    result_transh = ModelWrapper(model, training_factory)
    
else:
    
    result_transh = entrenar_transh(tripletas, embedding_dim=50, num_epochs=100)
    
    result_transh.save_to_directory(save_dir)




Found saved model. Loading...


In [15]:
sim_matrix, animes_validos, embeddings = calcular_similitudes_peliculas(
    result_transh,
    animes_interes
)

encontrar_peliculas_similares(result_transh, 'Cowboy_Bebop', top_k=3)


CÁLCULO DE SIMILITUDES CON EMBEDDINGS TransE

Matriz de similitud (coseno) entre 7 películas:

                    Cowboy_BebopRurouni_KenshinOh_My_GoddessAngel_SanctuaryZipang      Neon_Genesis_Evangelion_The_End_of_EvangelionInitial_D_Fourth_Stage
Cowboy_Bebop              1.0000      0.5404      0.4814      0.5274      0.5969      0.5894      0.3863
Rurouni_Kenshin           0.5404      1.0000      0.4161      0.4137      0.5168      0.4994      0.3614
Oh_My_Goddess             0.4814      0.4161      1.0000      0.4472      0.4932      0.5766      0.2884
Angel_Sanctuary           0.5274      0.4137      0.4472      1.0000      0.4687      0.3346      0.4295
Zipang                    0.5969      0.5168      0.4932      0.4687      1.0000      0.5492      0.2374
Neon_Genesis_Evangelion_The_End_of_Evangelion      0.5894      0.4994      0.5766      0.3346      0.5492      1.0000      0.1987
Initial_D_Fourth_Stage      0.3863      0.3614      0.2884      0.4295      0.2374      0.1987

In [ ]:
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
import numpy as np
import torch
from typing import List, Tuple

def entrenar_transR(triples: List[Tuple[str, str, str]], 
                              embedding_dim: int = 128,
                              num_epochs: int = 300,
                              genre_weight: int = 10):


    tf_all = TriplesFactory.from_labeled_triples(
        np.array(triples, dtype=str),
        create_inverse_triples=False
    )
    

    tf_train, tf_test = tf_all.split([0.8, 0.2], random_state=42)
    

    tf_train = TriplesFactory(
        mapped_triples=tf_train.mapped_triples,
        entity_to_id=tf_train.entity_to_id,
        relation_to_id=tf_train.relation_to_id,
        create_inverse_triples=True,
    )


    genre_relation_ids = [
        id for label, id in tf_train.relation_to_id.items()
        if ('type' in label or 'Genre' in label) and not label.endswith('_inverse')
    ]
    
    print(f"Identified {len(genre_relation_ids)} relation types for weighting.")

    train_tensor = tf_train.mapped_triples
    
    is_genre_mask = torch.isin(train_tensor[:, 1], torch.tensor(genre_relation_ids))
    genre_triples = train_tensor[is_genre_mask]
    
    if len(genre_triples) > 0:
        # Repetir los triples de género
        repeated_genres = genre_triples.repeat(genre_weight - 1, 1)
        new_train_tensor = torch.cat([train_tensor, repeated_genres], dim=0)
    else:
        new_train_tensor = train_tensor

    tf_train_weighted = TriplesFactory(
        mapped_triples=new_train_tensor,
        entity_to_id=tf_train.entity_to_id,
        relation_to_id=tf_train.relation_to_id, 
        create_inverse_triples=False,
    )

    print(f"Training Triples: {tf_train.num_triples} -> {tf_train_weighted.num_triples} (Weighted)")


    result = pipeline(
        model='TransR',
        training=tf_train_weighted,
        testing=tf_test,
        model_kwargs={
            'embedding_dim': embedding_dim,
            'scoring_fct_norm': 2
        },
        optimizer='Adam',
        optimizer_kwargs={'lr': 0.005},
        negative_sampler='bernoulli',
        negative_sampler_kwargs={'num_negs_per_pos': 50},
        training_kwargs={
            'num_epochs': num_epochs,
            'batch_size': 256
        },
        device='cpu',
        random_seed=42
    )
    
    return result

In [22]:
result_transR = entrenar_transR(tripletas, embedding_dim=100, num_epochs=100)

# Guardar el modelo
result_transR.save_to_directory("graph_embeddings_transR")

sim_matrix, animes_validos, embeddings = calcular_similitudes_peliculas(
    result_transR,
    animes_interes
)

encontrar_peliculas_similares(result_transR, 'Cowboy_Bebop', top_k=3)

INFO:pykeen.triples.splitting:done splitting triples to groups of sizes [22447, 9895]
INFO:pykeen.pipeline.api:Using device: cpu
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()


Identified 1 relation types for weighting.
Training Triples: 39580 -> 149668 (Weighted)


d:\Users\pedro\Documents\Programacion\Master\RRBC\venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Training epochs on cpu: 100%|██████████| 100/100 [2:47:55<00:00, 100.75s/epoch, loss=0.0589, prev_loss=0.0583] 
Evaluating on cpu:   0%|          | 0.00/9.89k [00:00<?, ?triple/s]WARNING:torch_max_mem.api:Encountered tensors on device_types={'cpu'} while only ['cuda'] are considered safe for automatic memory utilization maximization. This may lead to undocumented crashes (but can be safe, too).
Evaluating on cpu: 100%|██████████| 9.89k/9.89k [01:52<00:00, 87.9triple/s]
INFO:pykeen.evaluation.evaluator:Evaluation took 113.42s seconds
INFO:pykeen.triples.triples_factory:Stored TriplesFactory(num_entities=27252, num_relations=32, create_inverse_triples=False, num_triples=149668) to file:///D:/Users/pedro/Documents/Programacion/Master/RRBC


CÁLCULO DE SIMILITUDES CON EMBEDDINGS TransE

Matriz de similitud (coseno) entre 7 películas:

                    Cowboy_BebopRurouni_KenshinOh_My_GoddessAngel_SanctuaryZipang      Neon_Genesis_Evangelion_The_End_of_EvangelionInitial_D_Fourth_Stage
Cowboy_Bebop              1.0000      0.6991      0.5938      0.6351      0.5114      0.4745      0.5009
Rurouni_Kenshin           0.6991      1.0000      0.6477      0.6135      0.5372      0.2640      0.4029
Oh_My_Goddess             0.5938      0.6477      1.0000      0.5473      0.5021      0.3291      0.4353
Angel_Sanctuary           0.6351      0.6135      0.5473      1.0000      0.6332      0.3154      0.5072
Zipang                    0.5114      0.5372      0.5021      0.6332      1.0000      0.3363      0.6632
Neon_Genesis_Evangelion_The_End_of_Evangelion      0.4745      0.2640      0.3291      0.3154      0.3363      1.0000      0.3422
Initial_D_Fourth_Stage      0.5009      0.4029      0.4353      0.5072      0.6632      0.3422

# Conclusión tras la fase de validación (100 animes):

Tras haber entrenado los tres modelos (TransE, transH y TransR), con la ontología de demo (100 animes) llegamos a la siguiente conclusión:
- **TransE**:  Aunque ofrece resultados aceptables en esta demo, se considera un modelo demasiado simple para la ontología final. La documentación y la teoría sugieren que TransE tendrá dificultades para modelar relaciones complejas (1-a-N y N-a-N) cuando escalemos a 3000 animes, lo que probablemente resultará en una pérdida de precisión.
- **TransH**: Este modelo corrigió la "difusión" observada en TransE. Notamos una bajada general en las puntuaciones de similitud, lo cual es positivo: indica que el modelo está creando fronteras más estrictas entre conceptos y géneros, reduciendo los falsos positivos.
- **TransR** A pesar de la complejidad del modelo y el tamaño reducido del dataset, TransR ofreció el hallazgo más significativo: la conexión Cowboy Bebop -> Black Cat.
  - Mientras que TransE y TransH se guiaron por atributos explícitos (recomendando Gundam por compartir el género Mecha/Ciencia Ficción), TransR logró capturar una similitud semántica implícita.
  - Black Cat y Cowboy Bebop comparten temática (cazarrecompensas, estilo noir), aunque sus atributos de género en el grafo sean distintos.
  - Esto demuestra que TransR es capaz de proyectar entidades en espacios de relación específicos, separando eficazmente los atributos estructurales del "significado" o la ambientación de la obra. 